In [10]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [23]:
fbref_url = "https://fbref.com{}"
premier_league = fbref_url.format("/en/comps/9/Premier-League-Stats")
data = requests.get(premier_league)

## Retrieving Individual Player Data

In [796]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
import pickle
from IPython.display import clear_output


def make_request(endpoint: str) -> str:
    while True:
        response = requests.get(endpoint)
        if response.status_code == 429:  # Rate limited
            print("Rate limited. Pausing for 5 minutes...")
            print(endpoint)
            print(response.text)
            time.sleep(300)  # Wait for 5 minutes
        else:
            return response.text

def get_page_soup(url:str) -> BeautifulSoup:
    return BeautifulSoup(make_request(url), 'html.parser')

def extract_team_links(url:str) -> dict:
    league_teams_soup = get_page_soup(url)
    league_table = league_teams_soup.select('table.stats_table')[0]
    teams_dict = {team_link.get_text():fbref_url.format(team_link.get("href")) for team_link in league_table.find_all('a') if "squads" in team_link.get("href")}
    return teams_dict

def extract_player_links(url:str) -> dict:
    team_soup = get_page_soup(url)
    team_players_table = team_soup.select('table.stats_table')[0]
    players_links_dict = {
            l.get_text():fbref_url.format(l.get("href")) for l in team_players_table.find_all('a') 
            if (link := l.get("href")) and "players" in link and "matchlogs" not in link
        }
    return players_links_dict

def extract_player_details(url:str) -> dict:
    player_soup = get_page_soup(url)
    media_item_div = player_soup.select_one('div.media-item')
    image_src = media_item_div.find('img')['src'] if media_item_div else None

    # Extracting data dynamically
    for p in player_soup.find_all('p'):
        text = p.get_text(strip=True)
        # Extract position and footed
        if "Position:" in text:
            position_part = text.split("▪")[0].strip()
            footed_part = text.split("▪")[1].strip() if "▪" in text else None
            player_data['position'] = position_part.split(":")[1].strip()
            if footed_part:
                player_data['footed'] = footed_part.split(":")[1].strip()
        # Extract height and weight
        if "cm" in text and "kg" in text:
            height, weight = map(str.strip, text.split(',')[:2])
            player_data['height'] = height
            player_data['weight'] = weight.split('(')[0]
        # Extract date of birth and place of birth
        if "Born:" in text:
            birth_data = p.find("span", {"data-birth": True})
            if birth_data:
                player_data['dob'] = birth_data["data-birth"]
                birth_place = birth_data.find_next_sibling("span")
                if birth_place:
                    player_data['birth_place'] = birth_place.get_text(strip=True)[3:]  # Clean prefix if necessary
        # Extract national team
        if "National Team:" in text:
            player_data['national_team'] = text.split(":")[1].strip()
        # Extract club
        if "Club:" in text:
            player_data['club'] = text.split(":")[1].strip()
        # Extract wages and contract expiry
        if "Wages" in text:
            wages = p.select_one('.important').get_text(strip=True) if p.select_one('.important') else None
            player_data['wages'] = wages
            if "Expires" in text:
                contract_expires = text.split("Expires")[1].split(".")[0].strip()
                player_data['contract_expiry'] = contract_expires
    if image_src:
        player_data['img'] = image_src
    return player_data

# URLs
premier_league_links = [
    'https://fbref.com/en/comps/9/Premier-League-Stats',
    'https://fbref.com/en/comps/9/2023-2024/2023-2024-Premier-League-Stats',
    'https://fbref.com/en/comps/9/2022-2023/2022-2023-Premier-League-Stats',
    'https://fbref.com/en/comps/9/2021-2022/2021-t2022-Premier-League-Stats'
]

fbref_url = 'https://fbref.com{}'

# Load existing player list from CSV

all_players_dict = pd.read_csv('players.csv', index_col=0).to_dict(orient='index')

for premier_league_link in premier_league_links:
    # Fetch Premier League standings
    premier_league_year = premier_league_link.split('/')[-2]
    team_links = extract_team_links(premier_league_link)
    team_dict = {}
    for team_name, team_link in team_links.items():
        player_links = extract_player_links(team_link)
        team_players_dict = {}
        for player_name, player_link in player_links.items():
            if player_name in all_players_dict:
                print(f"{player_name} already in list. Skipping...")
                team_players_dict[player_name] = all_players_dict[player_name]
            else:
                player_data = extract_player_details(player_link)    
                team_players_dict[player_name] = player_data
                print(f"Retrieved player data from fbref {player_name}: {player_data}")
                all_players_dict[player_name] = player_data
                time.sleep(4)  
            
        clear_output(wait=True)
        team_dict[team_name] = team_players_dict
        print(squad_players_dict)
        time.sleep(5)
        clear_output(wait=True)
    pd.DataFrame.from_dict(all_players_dict).to_csv(f'players{premier_league_year }.csv', index=False)
    with open(f'team_dict_{premier_league_year}.pkl', 'wb') as f:
        pickle.dump(team_dict, f)

    print(f"Finished processing season: {premier_league_link.split('/')[-2]}")
    time.sleep(60)


Finished processing season: 9


KeyboardInterrupt: 

## Get All Matches

In [603]:
import io

def get_fixtures(url:str) -> BeautifulSoup:
    fixtures_soup = get_page_soup(url)
    fixtures_table = fixtures_soup.find(id='all_sched')
    return fixtures_table

def get_match_data(url:str) -> list:
    game_soup = get_page_soup(url)
    team_stats = list()
    team_game_stats = game_soup.find_all(id=re.compile("all_player_stats"))
    goaly_game_stats = game_soup.find_all(id=re.compile("keeper_stats"), class_ = "table_wrapper")
    for team_stat in team_game_stats:
        team_stats.append(pd.read_html(str(team_stat))[0])
    for count, goaly_stat in enumerate(goaly_game_stats):
        goaly_df = pd.read_html(str(goaly_stat))[0]
        team_stats[count] = [team_stats[count], goaly_df]
    return team_stats


matches_dict = dict()
for premiere_league_link in premier_league_links:
    premier_league_year = premier_league_link.split('/')[-2]
    fixtures = get_fixtures(premiere_league_link)
    match_report_links = [fbref_url.format(l.get('href')) for l in fixtures_table.find_all('a') if "Match Report" in l]
    match_reports = list()
    
    for count, match_link  in enumerate(match_report_links):
        print(f"getting match data for match: {link}. Progress: {((count+1)/len(match_report_links))*100}%")
        match_data = get_match_data(match_link)
        match_reports.append(match_data)
        time.sleep(4)
        clear_output()
    matches_dict[premier_league_year] = match_reports

In [604]:
(match_reports[0][0][1])

Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0 Unnamed: 3_level_0  \
              Player             Nation                Age                Min   
0        André Onana             cm CMR             28-136                 90   

  Shot Stopping                      Launched  ...  Passes        Goal Kicks  \
           SoTA GA Saves  Save% PSxG      Cmp  ... Launch% AvgLen        Att   
0             2  0     2  100.0  0.9        3  ...    34.5   29.6          0   

                 Crosses           Sweeper          
  Launch% AvgLen     Opp Stp  Stp%    #OPA AvgDist  
0     NaN    NaN      16   2  12.5       4    16.3  

[1 rows x 24 columns]

In [611]:
import pickle as pkl

with open('matches.pkl', 'wb') as f:
    pkl.dump(matches_dict, f)

In [ ]:
for premiere_

## Neo4j Setup

In [804]:
from neo4j import GraphDatabase
from neo4j.exceptions import ServiceUnavailable
import logging
import pandas as pd

NEO4J_URI='neo4j+s://2b521897.databases.neo4j.io'
NEO4J_USERNAME='neo4j'
NEO4J_PASSWORD='nCUihZktDGHjJUpQtKUal0IJhrk0eSuMdSFUm99Th3Q'
AURA_INSTANCEID='2b521897'
AURA_INSTANCENAME='Instance01'

In [803]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

In [677]:
# players_df = pd.read_csv('all_players_pd.csv')
print(players_data)




[{'Name': 'Virgil van Dijk', 'position': 'DF (CB, left)', 'footed': 'Right', 'height': '193cm', 'weight': '87kg', 'dob': '1991-07-08', 'birth_place': 'Breda, Netherlands', 'national_team': 'Netherlandsnl', 'club': 'Liverpool', 'wages': '￡ 220,000 Weekly', 'contract_expiry': 'June 2025', 'img': 'https://fbref.com/req/202302030/images/headshots/e06683ca_2022.jpg'}, {'Name': 'Mohamed Salah', 'position': 'FW-MF (AM-WM, right)', 'footed': 'Left', 'height': '175cm', 'weight': '71kg', 'dob': '1992-06-15', 'birth_place': 'Basyūn, Egypt', 'national_team': 'Egypteg', 'club': 'Liverpool', 'wages': '￡ 350,000 Weekly', 'contract_expiry': 'June 2025', 'img': 'https://fbref.com/req/202302030/images/headshots/e342ad68_2022.jpg'}, {'Name': 'Ryan Gravenberch', 'position': 'MF (CM-DM-WM)', 'footed': 'Right', 'height': '190cm', 'weight': '78kg', 'dob': '2002-05-16', 'birth_place': 'Amsterdam, Netherlands', 'national_team': 'Netherlandsnl', 'club': 'Liverpool', 'wages': '￡ 150,000 Weekly', 'contract_expiry

In [721]:
fixtures_2024 = get_fixtures('https://fbref.com/en/comps/9/schedule/Premier-League-Scores-and-Fixtures')
fixtures_df = pd.read_html(str(fixtures_2024))[0]
fixtures_df = fixtures_df.rename(columns={'Wk':'GW'})
fixtures_df = fixtures_df[~fixtures_df['GW'].isnull()]
fixtures_df['GW'] = "GW"+fixtures_df['GW'].astype(int).astype(str)

/var/folders/78/b2j3spl172n6y7p_sw2jggr40000gn/T/ipykernel_54417/760373454.py:2: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  fixtures_df = pd.read_html(str(fixtures_2024))[0]


In [728]:
fixtures_df[['Home_Score', 'Away_Score']] = fixtures_df['Score'].str.split('–', expand=True)

In [813]:
fixtures_df

,GW,Day,Date,Time,Home,xG,Score,xG.1,Away,Attendance,Venue,Referee,Match Report,Notes,Home_Score,Away_Score
0,GW1,Fri,2024-08-16,20:00,Manchester Utd,2.4,1–0,0.4,Fulham,73297.0,Old Trafford,Robert Jones,Match Report,NaN,1,0
1,GW1,Sat,2024-08-17,12:30,Ipswich Town,0.5,0–2,2.6,Liverpool,30014.0,Portman Road Stadium,Tim Robinson,Match Report,NaN,0,2
2,GW1,Sat,2024-08-17,15:00,Newcastle Utd,0.3,1–0,1.8,Southampton,52196.0,St James' Park,Craig Pawson,Match Report,NaN,1,0
3,GW1,Sat,2024-08-17,15:00,Nott'ham Forest,1.3,1–1,1.2,Bournemouth,29763.0,The City Ground,Michael Oliver,Match Report,NaN,1,1
4,GW1,Sat,2024-08-17,15:00,Everton,0.5,0–3,1.4,Brighton,39217.0,Goodison Park,Simon Hooper,Match Report,NaN,0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,GW38,Sun,2025-05-25,16:00,Fulham,NaN,NaN,NaN,Manchester City,NaN,Craven Cottage,NaN,Head-to-Head,NaN,NaN,NaN
414,GW38,Sun,2025-05-25,16:00,Nott'ham Forest,NaN,NaN,NaN,Chelsea,NaN,The City Ground,NaN,Head-to-Head,NaN,NaN,NaN
415,GW38,Sun,2025-05-25,16:00,Manchester Utd,NaN,NaN,NaN,Aston Villa,NaN,Old Trafford,NaN,Head-to-Head,NaN,NaN,NaN
416,GW38,Sun,2025-05-25,16:00,Wolves,NaN,NaN,NaN,Brentford,NaN,Molineux Stadium,NaN,Head-to-Head,NaN,NaN,NaN


In [839]:
match_reports[0][1][0]

Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0  \
               Player                  #             Nation   
0       Rodrigo Muniz                9.0             br BRA   
1        Raúl Jiménez                7.0             mx MEX   
2          Alex Iwobi               17.0             ng NGA   
3        Adama Traoré               11.0             es ESP   
4        Harry Wilson                8.0            wls WAL   
5    Emile Smith Rowe               32.0            eng ENG   
6         Tom Cairney               10.0            sct SCO   
7          Saša Lukić               20.0             rs SRB   
8      Jay Stansfield               28.0            eng ENG   
9     Andreas Pereira               18.0             br BRA   
10      Harrison Reed                6.0            eng ENG   
11   Antonee Robinson               33.0             us USA   
12      Calvin Bassey                3.0             ng NGA   
13          Issa Diop               31.0             fr FRA   
14         Kenny Tete                2.0             nl NED   
15         Bernd Leno                1.0             de GER   
16         16 Players                NaN                NaN   

   Unnamed: 3_level_0 Unnamed: 4_level_0 Unnamed: 5_level_0 Performance      \
                  Pos                Age                Min         Gls Ast   
0                  FW             23-104                 77           0   0   
1                  FW             33-103                 13           0   0   
2                  LW             28-105                 90           0   0   
3                  RW             28-204                 77           0   0   
4                  RW             27-147                 13           0   0   
5                  AM             24-019                 63           0   0   
6               DM,AM             33-209                 27           0   0   
7                  DM             28-003                 89           0   0   
8               AM,DM             21-266                  1           0   0   
9               DM,AM             28-228                 89           0   0   
10                 DM             29-202                  1           0   0   
11                 LB             27-008                 90           0   0   
12                 CB             24-229                 90           0   0   
13                 CB             27-220                 90           0   0   
14                 RB             28-312                 90           0   0   
15                 GK             32-165                 90           0   0   
16                NaN                NaN                990           0   0   

             ... SCA     Passes                  Carries      Take-Ons       
   PK PKatt  ... SCA GCA    Cmp  Att   Cmp% PrgP Carries PrgC      Att Succ  
0   0     0  ...   2   0      5    6   83.3    1      11    0        1    1  
1   0     0  ...   0   0      2    4   50.0    0       2    1        0    0  
2   0     0  ...   0   0     19   29   65.5    2      31    1        5    1  
3   0     0  ...   4   0     14   21   66.7    0      25    5        9    4  
4   0     0  ...   0   0      5    6   83.3    0       6    1        0    0  
5   0     0  ...   0   0     20   27   74.1    1      23    0        0    0  
6   0     0  ...   1   0     15   17   88.2    4      14    2        3    2  
7   0     0  ...   0   0     40   45   88.9    4      30    1        0    0  
8   0     0  ...   0   0      0    0    NaN    0       0    0        0    0  
9   0     0  ...   9   0     37   55   67.3    5      28    1        2    0  
10  0     0  ...   0   0      1    1  100.0    0       1    0        0    0  
11  0     0  ...   0   0     32   42   76.2    2      31    2        2    0  
12  0     0  ...   1   0     57   62   91.9    5      51    1        0    0  
13  0     0  ...   0   0     28   35   80.0    2      26    0        0    0  
14  0     0  ...   2   0     24   32   75.0    2      2

In [735]:
# 
# (LEAGUE: PremLeague24-25) -> GWs -> Match -> Has_Home/Away
# 
# 


In [805]:
with driver.session(database="neo4j") as session:
    query = "CREATE (l:League {name: $name})"
    session.run(query, {"name":"PremierLegue24-25"})

In [808]:
team_players = dict()
player_details = list()
for team, players in team_dict.items():
    team_players[team] = list()
    for player in players.keys():
        team_players[team].append(player)
        players[player]['Name'] = player
        player_details.append(players[player])
    team_players[team] = current_team_players

In [809]:
with driver.session(database="neo4j") as session:
    query = """
            UNWIND $players AS player
            MERGE (p:Player {name: player.Name})
            SET p.position = player.position,
                p.footed = player.footed,
                p.height = player.height,
                p.weight = player.weight,
                p.dob = player.dob,
                p.birth_place = player.birth_place,
                p.national_team = player.national_team,
                p.club = player.club,
                p.wages = player.wages,
                p.contract_expiry = player.contract_expiry,
                p.img = player.img
    """
    
    session.run(query, players=player_details)

In [827]:
with driver.session(database="neo4j") as session:
    league_name = "PremierLeague24-25"

    # Iterate through each row to create nodes and relationships
    for _, row in fixtures_df.iterrows():
        # Create or link GameWeek
        gameweek = row['GW']
        session.run(
            """
            MATCH (l:League {name: $league_name})
            MERGE (gw:GameWeek {name: $gameweek})
            MERGE (l)-[:HAS_GAME_WEEK]->(gw)
            """,
            league_name=league_name,
            gameweek=gameweek
        )
        
        # Create Match Node
        session.run(
            """
            MATCH (gw:GameWeek {name: $gameweek})
            MERGE (m:Match {date: $date, time: $time, home: $home, away: $away, venue: $venue})
            MERGE (gw)-[:HAS_MATCH]->(m)
            """,
            gameweek=gameweek,
            date=row['Date'],
            time=row['Time'],
            home=row['Home'],
            away=row['Away'],
            venue=row['Venue']
        )
        
        # Create Home and Away Score Nodes
        session.run(
            """
            MATCH (m:Match {date: $date, home: $home, away: $away})
            MERGE (hs:Score {value: $home_score})
            MERGE (as:Score {value: $away_score})
            MERGE (m)-[:HAS_HOME_SCORE]->(hs)
            MERGE (m)-[:HAS_AWAY_SCORE]->(as)
            """,
            date=row['Date'],
            home=row['Home'],
            away=row['Away'],
            home_score=row['Home_Score'],
            away_score=row['Away_Score']
        )
        
        # Create Referee Node
        session.run(
            """
            MATCH (m:Match {date: $date, home: $home, away: $away})
            MERGE (r:Referee {name: $referee})
            MERGE (m)-[:REFEREED_BY]->(r)
            """,
            date=row['Date'],
            home=row['Home'],
            away=row['Away'],
            referee=row['Referee']
        )

KeyboardInterrupt: 

In [832]:
all_players_dict['Omari Hutchinson']

{'position': 'FW-MF',
 'footed': 'Both',
 'height': nan,
 'weight': nan,
 'dob': '2003-10-30',
 'birth_place': 'England, United Kingdom',
 'national_team': 'EnglandengOther',
 'club': 'Ipswich Town',
 'wages': nan,
 'contract_expiry': nan,
 'img': 'https://fbref.com/req/202302030/images/headshots/bd9553e6_2022.jpg',
 'Name': 'Omari Hutchinson'}

In [841]:
# clean up before sending
with open('match_reports.pkl','rb') as f:
    d = pkl.load(f)

[[    Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0  \
                  Player                  #             Nation   
  0      Bruno Fernandes                8.0             pt POR   
  1      Marcus Rashford               10.0            eng ENG   
  2          Amad Diallo               16.0             ci CIV   
  3   Alejandro Garnacho               17.0             ar ARG   
  4          Mason Mount                7.0            eng ENG   
  5       Joshua Zirkzee               11.0             nl NED   
  6        Kobbie Mainoo               37.0            eng ENG   
  7      Scott McTominay               39.0            sct SCO   
  8             Casemiro               18.0             br BRA   
  9          Diogo Dalot               20.0             pt POR   
  10   Lisandro Martínez                6.0             ar ARG   
  11       Harry Maguire                5.0            eng ENG   
  12         Jonny Evans               35.0            nir NIR   
  13   Nou